# Sistema experto MULC · Evaluación previa de conformidad cambiaria

Sistema experto **basado en reglas** (biblioteca **Experta**, encadenamiento hacia adelante) que asiste a responsables de comercio exterior y tesorería pyme en la evaluación previa de una solicitud de acceso al **MULC** del BCRA. Determina si la operación es **AUTORIZADA**, **CONDICIONAL / requiere gestión adicional** o **RECHAZADA / bloqueada**, mostrando siempre la **traza de reglas** que fundamenta el dictamen.

> Es un asistente decisional analítico y de control previo. **No constituye asesoramiento legal vinculante.**

### Cómo ejecutarlo en Visual Studio Code
1. Abrí este `.ipynb` en VS Code (extensión **Jupyter** + **Python** instaladas).
2. Elegí un kernel de Python 3.9 o superior (probado en 3.12) con **Select Kernel**.
3. Ejecutá las celdas en orden (**Run All**). Las celdas `%%writefile` crean en esta carpeta los tres archivos del proyecto: `motor_mulc.py`, `casos_prueba.py` y `app_streamlit.py`.
4. Para la interfaz web, en la terminal de VS Code: `streamlit run app_streamlit.py`.

### Contenido
1. Instalación · 2. Diseño del conocimiento · 3. Motor de inferencia · 4. Ejemplo con traza · 5. Casos de prueba · 6. Interfaz Streamlit · 7. Parámetros, supuestos y límites

## 1. Instalación de dependencias

In [39]:
import sys, subprocess, importlib.util

for paquete in ("experta", "streamlit"):
    if importlib.util.find_spec(paquete) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", paquete])

print("Python", sys.version.split()[0])
print("experta:", "OK" if importlib.util.find_spec("experta") else "FALTA")
print("streamlit:", "OK" if importlib.util.find_spec("streamlit") else "FALTA")

Python 3.14.1
experta: OK
streamlit: OK


> **Nota de compatibilidad.** Experta 1.9.x depende de `frozendict 1.2`, que usa `collections.Mapping` (eliminado en Python 3.10+). El módulo `motor_mulc.py` restituye esos alias **antes** de importar Experta, por lo que no hay que tocar nada a mano.

## 2. Diseño del conocimiento

### 2.1 Hechos de entrada (nivel 0)
| Entrada | Hecho | Valores |
|---|---|---|
| Tipo de solicitud | `tipo_solicitud` | `importacion_bienes`, `servicios`, `pasivos_financieros` |
| Historial bursátil | `opero_mep` | Sí / No (MEP/CCL en la ventana de inhibición) |
| Plazo desde registro aduanero / factura | `dias_plazo` | entero ≥ 0 (días corridos) |
| Categoría del bien | `categoria` | `bienes_capital`, `insumos_criticos`, `consumo_general` |
| DJ de activos externos | `activos_externos` | Sí / No (disponibilidad líquida > USD 100.000 no aplicada) |
| Vinculación de la contraparte | `vinculada` | Sí / No |

Un parámetro **no informado** se declara como hecho `Faltante`, lo que habilita el manejo de casos incompletos (RF-06).

### 2.2 Encadenamiento hacia adelante en 4 niveles (RF-04)
```
Nivel 0  Hechos iniciales (formulario)         Dato · Faltante
   │
Nivel 1  Hechos intermedios básicos            inhibicion_cruzada_activa · plazo_minimo_requerido
   │                                           dj_activos_en_regla · requiere_conformidad_previa · faltante      R01–R17
Nivel 2  Hechos intermedios compuestos         plazo_exigible_valido · causal_bloqueante_vigente
   │                                           estado_documental · pago_a_la_vista_habilitado · condicionante   R18–R25
Nivel 3  Situación regulatoria                 impedida · condicionada · sin_obstaculos · indeterminada          R26–R30
   │
Nivel 4  Regla resolutiva ESTADO_OPERACION     RECHAZADO · CONDICIONAL · AUTORIZADO · SIN_CONCLUSION             R31–R34
```
**Precedencia:** una causal bloqueante (MEP/CCL en la ventana) prevalece sobre cualquier otra y no depende del bien. Si no hay bloqueo, cualquier condicionante cierto (plazo, fondos propios, conformidad previa) da dictamen **condicional**. Solo cuando faltan datos y **no** hay un veredicto cierto se emite *"Información insuficiente para emitir dictamen"*.

### 2.3 Cobertura de requisitos funcionales
| RF | Prioridad | Dónde se cubre |
|---|---|---|
| RF-01 Incompatibilidad bursátil | Must | R01, R02, R21, R26, R31 |
| RF-02 Clasificación de pago de bienes | Must | R03–R06, R18–R20, R22, R27 |
| RF-03 DJ y activos externos líquidos | Must | R07, R08, R23, R24, R28 |
| RF-04 Inferencia en múltiples niveles | Must | 4 niveles, R01–R34 |
| RF-05 Interfaz y explicabilidad | Must | `app_streamlit.py` + traza de cada evaluación |
| RF-06 Casos incompletos | Should | R12–R17, R30, R34 |
| RF-07 Contrapartes vinculadas | Could | R09–R11, R25, R28 |

## 3. Motor de inferencia (base de conocimiento)

La celda siguiente escribe `motor_mulc.py` con: parámetros normativos aislados, catálogo de reglas con su fundamento, las **34 reglas** (`@Rule` de Experta) y la función `evaluar(...)`. No hay cascadas de `if-else`: la decisión surge de la agenda del motor.

In [40]:
%%writefile motor_mulc.py
"""
Sistema experto MULC - Motor de inferencia (Experta, encadenamiento hacia adelante)

Asiste en la evaluación PREVIA de conformidad regulatoria antes de cursar una
solicitud de acceso al MULC del BCRA. No es asesoramiento legal vinculante.

Arquitectura:
    Nivel 0  Hechos iniciales (Dato)         <- formulario del usuario
    Nivel 1  Hechos intermedios básicos      <- inhibición, plazo mínimo, DJ, vinculación, faltantes
    Nivel 2  Hechos intermedios compuestos   <- plazo exigible válido, causal bloqueante, estado documental
    Nivel 3  Situación regulatoria           <- impedida / condicionada / sin obstáculos / indeterminada
    Nivel 4  Regla resolutiva                <- ESTADO_OPERACION (dictamen final)
"""

# --------------------------------------------------------------------------
# Compatibilidad: Experta 1.9.x depende de frozendict 1.2, que usa
# collections.Mapping (eliminado en Python 3.10+). Se restituyen los alias
# ANTES de importar experta.
# --------------------------------------------------------------------------
import collections
import collections.abc

for _nombre in ("Mapping", "MutableMapping", "Iterable", "Callable",
                "Sequence", "MutableSequence", "Set", "MutableSet"):
    if not hasattr(collections, _nombre):
        setattr(collections, _nombre, getattr(collections.abc, _nombre))

from experta import KnowledgeEngine, Fact, Rule, MATCH, TEST, NOT, P  # noqa: E402

# --------------------------------------------------------------------------
# PARÁMETROS NORMATIVOS (premisas aisladas y ajustables)
# Al cambiar la norma se modifica SOLO este bloque, sin reentrenar nada.
# Valores simplificados a efectos del trabajo: validar contra el texto
# ordenado vigente de "Exterior y Cambios" del BCRA antes de uso real.
# --------------------------------------------------------------------------
VENTANA_MEP_DIAS = 90            # días corridos de inhibición por operar MEP/CCL
UMBRAL_ACTIVOS_USD = 100_000     # umbral de libre disponibilidad de activos externos
PLAZOS_MINIMOS = {               # días mínimos de diferimiento por categoría
    "bienes_capital": 30,
    "insumos_criticos": 0,       # 0 = habilitado a la vista
    "consumo_general": 60,
}

TIPOS = {
    "importacion_bienes": "Pago diferido de importaciones de bienes",
    "servicios": "Pago de servicios",
    "pasivos_financieros": "Cancelación de pasivos financieros",
}
CATEGORIAS = {
    "bienes_capital": "Bienes de Capital",
    "insumos_criticos": "Insumos Críticos de Salud/Energía",
    "consumo_general": "Bienes de Consumo General",
}
TIPOS_SERVICIO_DEUDA = ("servicios", "pasivos_financieros")

ETIQUETAS_DICTAMEN = {
    "AUTORIZADO": "AUTORIZADO",
    "CONDICIONAL": "CONDICIONAL / REQUIERE GESTIÓN ADICIONAL",
    "RECHAZADO": "RECHAZADO / BLOQUEADO",
    "SIN_CONCLUSION": "SIN CONCLUSIÓN DETERMINABLE / FALTAN PARÁMETROS",
}


def _usd(valor):
    return f"{valor:,}".replace(",", ".")


ETIQUETAS_DATOS = {
    "tipo_solicitud": "Tipo de solicitud",
    "opero_mep": f"Operó MEP/CCL en los últimos {VENTANA_MEP_DIAS} días",
    "dias_plazo": "Días corridos desde registro aduanero / factura",
    "categoria": "Categoría del bien",
    "activos_externos": f"Disponibilidad líquida en el exterior > USD {_usd(UMBRAL_ACTIVOS_USD)} no aplicada",
    "vinculada": "Contraparte vinculada / mismo grupo económico",
}

# --------------------------------------------------------------------------
# CATÁLOGO DE REGLAS (documentación y trazabilidad normativa)
# --------------------------------------------------------------------------
N_INCOMP = "BCRA - Exterior y Cambios: incompatibilidad cruzada por operaciones MEP/CCL"
N_PLAZOS = "BCRA - Exterior y Cambios: plazos de acceso al MULC para pagos de importaciones (Com. 'A' 7770, 'A' 7782, 'A' 7840 y concordantes)"
N_DJ = "BCRA - Exterior y Cambios: declaración jurada de activos externos líquidos"
N_VINC = "BCRA - Exterior y Cambios: pagos a contrapartes vinculadas (conformidad previa)"
N_INT = "Derivación interna del motor (hecho intermedio)"
N_RES = "Regla resolutiva del sistema (ESTADO_OPERACION)"

CATALOGO = {
    # ---------------- Nivel 1 ----------------
    "R01": dict(nivel=1, rf="RF-01", norma=N_INCOMP,
                regla=f"SI operó MEP/CCL en los últimos {VENTANA_MEP_DIAS} días ENTONCES inhibicion_cruzada_activa = Sí"),
    "R02": dict(nivel=1, rf="RF-01", norma=N_INCOMP,
                regla=f"SI NO operó MEP/CCL en los últimos {VENTANA_MEP_DIAS} días ENTONCES inhibicion_cruzada_activa = No"),
    "R03": dict(nivel=1, rf="RF-02", norma=N_PLAZOS,
                regla="SI importación Y categoría = Bienes de Capital ENTONCES plazo_minimo_requerido = plazo BK"),
    "R04": dict(nivel=1, rf="RF-02", norma=N_PLAZOS,
                regla="SI importación Y categoría = Insumos Críticos ENTONCES plazo_minimo_requerido = a la vista"),
    "R05": dict(nivel=1, rf="RF-02", norma=N_PLAZOS,
                regla="SI importación Y categoría = Consumo General ENTONCES plazo_minimo_requerido = plazo consumo"),
    "R06": dict(nivel=1, rf="RF-02", norma=N_INT,
                regla="SI la solicitud es de servicios o pasivos financieros ENTONCES plazo_no_aplica = Sí"),
    "R07": dict(nivel=1, rf="RF-03", norma=N_DJ,
                regla=f"SI posee activos externos líquidos > USD {_usd(UMBRAL_ACTIVOS_USD)} no aplicados ENTONCES dj_activos_en_regla = No"),
    "R08": dict(nivel=1, rf="RF-03", norma=N_DJ,
                regla=f"SI NO posee activos externos líquidos > USD {_usd(UMBRAL_ACTIVOS_USD)} no aplicados ENTONCES dj_activos_en_regla = Sí"),
    "R09": dict(nivel=1, rf="RF-07", norma=N_VINC,
                regla="SI servicios/pasivos Y contraparte vinculada ENTONCES requiere_conformidad_previa = Sí"),
    "R10": dict(nivel=1, rf="RF-07", norma=N_VINC,
                regla="SI servicios/pasivos Y contraparte NO vinculada ENTONCES requiere_conformidad_previa = No"),
    "R11": dict(nivel=1, rf="RF-07", norma=N_INT,
                regla="SI importación de bienes ENTONCES requiere_conformidad_previa = No (no aplica)"),
    "R12": dict(nivel=1, rf="RF-06", norma=N_INT,
                regla="SI falta el tipo de solicitud ENTONCES faltante = tipo de solicitud"),
    "R13": dict(nivel=1, rf="RF-06", norma=N_INT,
                regla="SI falta el historial bursátil MEP/CCL ENTONCES faltante = historial bursátil"),
    "R14": dict(nivel=1, rf="RF-06", norma=N_INT,
                regla="SI importación Y falta el plazo desde registro/factura ENTONCES faltante = plazo"),
    "R15": dict(nivel=1, rf="RF-06", norma=N_INT,
                regla="SI importación Y falta la categoría del bien ENTONCES faltante = categoría"),
    "R16": dict(nivel=1, rf="RF-06", norma=N_INT,
                regla="SI falta la declaración jurada de activos externos ENTONCES faltante = DJ de activos externos"),
    "R17": dict(nivel=1, rf="RF-06", norma=N_INT,
                regla="SI servicios/pasivos Y falta la vinculación de la contraparte ENTONCES faltante = vinculación"),
    # ---------------- Nivel 2 ----------------
    "R18": dict(nivel=2, rf="RF-02", norma=N_INT,
                regla="SI plazo_no_aplica ENTONCES plazo_exigible_valido = Sí"),
    "R19": dict(nivel=2, rf="RF-02", norma=N_PLAZOS,
                regla="SI días transcurridos >= plazo_minimo_requerido ENTONCES plazo_exigible_valido = Sí"),
    "R20": dict(nivel=2, rf="RF-02", norma=N_PLAZOS,
                regla="SI días transcurridos < plazo_minimo_requerido ENTONCES plazo_exigible_valido = No y se agrega condicionante de plazo"),
    "R21": dict(nivel=2, rf="RF-01", norma=N_INCOMP,
                regla="SI inhibicion_cruzada_activa ENTONCES causal_bloqueante_vigente = Sí"),
    "R22": dict(nivel=2, rf="RF-02", norma=N_PLAZOS,
                regla="SI importación de insumos críticos Y DJ en regla Y sin inhibición ENTONCES pago_a_la_vista_habilitado = Sí"),
    "R23": dict(nivel=2, rf="RF-03", norma=N_INT,
                regla="SI dj_activos_en_regla Y NO requiere conformidad previa ENTONCES estado_documental = completo"),
    "R24": dict(nivel=2, rf="RF-03", norma=N_DJ,
                regla="SI dj_activos_en_regla = No ENTONCES estado_documental = pendiente y condicionante de fondos propios"),
    "R25": dict(nivel=2, rf="RF-07", norma=N_VINC,
                regla="SI requiere_conformidad_previa ENTONCES estado_documental = pendiente y condicionante de conformidad BCRA"),
    # ---------------- Nivel 3 ----------------
    "R26": dict(nivel=3, rf="RF-04", norma=N_INCOMP,
                regla="SI causal_bloqueante_vigente ENTONCES situacion_regulatoria = impedida"),
    "R27": dict(nivel=3, rf="RF-04", norma=N_PLAZOS,
                regla="SI sin inhibición Y plazo_exigible_valido = No ENTONCES situacion_regulatoria = condicionada"),
    "R28": dict(nivel=3, rf="RF-04", norma=N_INT,
                regla="SI sin inhibición Y estado_documental = pendiente Y plazo no incumplido ENTONCES situacion_regulatoria = condicionada"),
    "R29": dict(nivel=3, rf="RF-04", norma=N_INT,
                regla="SI sin inhibición Y plazo válido Y estado_documental = completo ENTONCES situacion_regulatoria = sin obstáculos"),
    "R30": dict(nivel=3, rf="RF-06", norma=N_INT,
                regla="SI hay parámetros faltantes Y no se derivó otra situación ENTONCES situacion_regulatoria = indeterminada"),
    # ---------------- Nivel 4 ----------------
    "R31": dict(nivel=4, rf="RF-01", norma=N_RES,
                regla="SI situacion_regulatoria = impedida ENTONCES ESTADO_OPERACION = RECHAZADO / BLOQUEADO"),
    "R32": dict(nivel=4, rf="RF-04", norma=N_RES,
                regla="SI situacion_regulatoria = condicionada ENTONCES ESTADO_OPERACION = CONDICIONAL / REQUIERE GESTIÓN ADICIONAL"),
    "R33": dict(nivel=4, rf="RF-04", norma=N_RES,
                regla="SI situacion_regulatoria = sin obstáculos ENTONCES ESTADO_OPERACION = AUTORIZADO"),
    "R34": dict(nivel=4, rf="RF-06", norma=N_RES,
                regla="SI situacion_regulatoria = indeterminada ENTONCES ESTADO_OPERACION = SIN CONCLUSIÓN DETERMINABLE"),
}

# Prioridades (salience): el motor procesa nivel por nivel; la regla de
# "información insuficiente" solo dispara cuando ninguna otra puede hacerlo.
S_N1, S_N2, S_N3, S_N4, S_INDETERMINADA = 40, 30, 20, 10, -10


# --------------------------------------------------------------------------
# HECHOS
# --------------------------------------------------------------------------
class Dato(Fact):
    """Hecho de entrada (nivel 0): nombre, valor."""


class Faltante(Fact):
    """Hecho de entrada (nivel 0): parámetro NO informado por el usuario (RF-06)."""


class Hecho(Fact):
    """Hecho derivado por el motor (niveles 1 a 3): nombre, valor."""


class Dictamen(Fact):
    """Hecho final (nivel 4): estado."""


# --------------------------------------------------------------------------
# MOTOR
# --------------------------------------------------------------------------
class MotorMULC(KnowledgeEngine):

    def __init__(self):
        super().__init__()
        self.traza = []
        self.resultado = None

    # ----- utilidades (no contienen lógica de negocio) -----
    def _disparar(self, rid, detalle=""):
        info = CATALOGO[rid]
        self.traza.append({
            "orden": len(self.traza) + 1, "regla": rid, "nivel": info["nivel"],
            "rf": info["rf"], "descripcion": info["regla"], "detalle": detalle,
            "norma": info["norma"],
        })

    def _valores(self, nombre):
        return [f["valor"] for f in self.facts.values()
                if type(f).__name__ == "Hecho" and f.get("nombre") == nombre]

    def _cerrar(self, estado, mensaje, motivos):
        self.declare(Dictamen(estado=estado))
        self.resultado = {"estado": estado, "etiqueta": ETIQUETAS_DICTAMEN[estado],
                          "mensaje": mensaje, "motivos": motivos}

    # ======================= NIVEL 1 =======================
    @Rule(Dato(nombre="opero_mep", valor=True), salience=S_N1)
    def r01_inhibicion_activa(self):
        self.declare(Hecho(nombre="inhibicion_cruzada_activa", valor=True))
        self._disparar("R01", "Dato ingresado: operó MEP/CCL = Sí")

    @Rule(Dato(nombre="opero_mep", valor=False), salience=S_N1)
    def r02_inhibicion_inactiva(self):
        self.declare(Hecho(nombre="inhibicion_cruzada_activa", valor=False))
        self._disparar("R02", "Dato ingresado: operó MEP/CCL = No")

    @Rule(Dato(nombre="tipo_solicitud", valor="importacion_bienes"),
          Dato(nombre="categoria", valor="bienes_capital"), salience=S_N1)
    def r03_plazo_bienes_capital(self):
        m = PLAZOS_MINIMOS["bienes_capital"]
        self.declare(Hecho(nombre="plazo_minimo_requerido", valor=m))
        self._disparar("R03", f"Plazo mínimo para Bienes de Capital: {m} días")

    @Rule(Dato(nombre="tipo_solicitud", valor="importacion_bienes"),
          Dato(nombre="categoria", valor="insumos_criticos"), salience=S_N1)
    def r04_plazo_insumos_criticos(self):
        m = PLAZOS_MINIMOS["insumos_criticos"]
        self.declare(Hecho(nombre="plazo_minimo_requerido", valor=m))
        self._disparar("R04", f"Plazo mínimo para Insumos Críticos: {m} días (a la vista)")

    @Rule(Dato(nombre="tipo_solicitud", valor="importacion_bienes"),
          Dato(nombre="categoria", valor="consumo_general"), salience=S_N1)
    def r05_plazo_consumo_general(self):
        m = PLAZOS_MINIMOS["consumo_general"]
        self.declare(Hecho(nombre="plazo_minimo_requerido", valor=m))
        self._disparar("R05", f"Plazo mínimo para Bienes de Consumo General: {m} días")

    @Rule(Dato(nombre="tipo_solicitud", valor=P(lambda t: t in TIPOS_SERVICIO_DEUDA)), salience=S_N1)
    def r06_plazo_no_aplica(self):
        self.declare(Hecho(nombre="plazo_no_aplica", valor=True))
        self._disparar("R06", "El plazo de importaciones no aplica a servicios / pasivos financieros")

    @Rule(Dato(nombre="activos_externos", valor=True), salience=S_N1)
    def r07_dj_no_en_regla(self):
        self.declare(Hecho(nombre="dj_activos_en_regla", valor=False))
        self._disparar("R07", "Dato ingresado: posee activos externos líquidos sobre el umbral")

    @Rule(Dato(nombre="activos_externos", valor=False), salience=S_N1)
    def r08_dj_en_regla(self):
        self.declare(Hecho(nombre="dj_activos_en_regla", valor=True))
        self._disparar("R08", "Dato ingresado: no posee activos externos líquidos sobre el umbral")

    @Rule(Dato(nombre="tipo_solicitud", valor=P(lambda t: t in TIPOS_SERVICIO_DEUDA)),
          Dato(nombre="vinculada", valor=True), salience=S_N1)
    def r09_requiere_conformidad(self):
        self.declare(Hecho(nombre="requiere_conformidad_previa", valor=True))
        self._disparar("R09", "Contraparte vinculada en servicios / pasivos financieros")

    @Rule(Dato(nombre="tipo_solicitud", valor=P(lambda t: t in TIPOS_SERVICIO_DEUDA)),
          Dato(nombre="vinculada", valor=False), salience=S_N1)
    def r10_no_requiere_conformidad(self):
        self.declare(Hecho(nombre="requiere_conformidad_previa", valor=False))
        self._disparar("R10", "Contraparte no vinculada")

    @Rule(Dato(nombre="tipo_solicitud", valor="importacion_bienes"), salience=S_N1)
    def r11_conformidad_no_aplica(self):
        self.declare(Hecho(nombre="requiere_conformidad_previa", valor=False))
        self._disparar("R11", "La conformidad por vinculación se evalúa solo en servicios / pasivos")

    # ----- faltantes (RF-06) -----
    @Rule(Faltante(nombre="tipo_solicitud"), salience=S_N1)
    def r12_falta_tipo(self):
        self.declare(Hecho(nombre="faltante", valor="Tipo de solicitud"))
        self._disparar("R12", "No se informó el tipo de solicitud")

    @Rule(Faltante(nombre="opero_mep"), salience=S_N1)
    def r13_falta_historial(self):
        self.declare(Hecho(nombre="faltante", valor="Historial bursátil (MEP/CCL)"))
        self._disparar("R13", "No se informó el historial bursátil")

    @Rule(Dato(nombre="tipo_solicitud", valor="importacion_bienes"),
          Faltante(nombre="dias_plazo"), salience=S_N1)
    def r14_falta_plazo(self):
        self.declare(Hecho(nombre="faltante", valor="Plazo desde registro aduanero / factura"))
        self._disparar("R14", "No se informó el plazo transcurrido")

    @Rule(Dato(nombre="tipo_solicitud", valor="importacion_bienes"),
          Faltante(nombre="categoria"), salience=S_N1)
    def r15_falta_categoria(self):
        self.declare(Hecho(nombre="faltante", valor="Categoría del bien"))
        self._disparar("R15", "No se informó la categoría del bien")

    @Rule(Faltante(nombre="activos_externos"), salience=S_N1)
    def r16_falta_dj(self):
        self.declare(Hecho(nombre="faltante", valor="Declaración jurada de activos externos"))
        self._disparar("R16", "No se informó la declaración jurada de activos externos")

    @Rule(Dato(nombre="tipo_solicitud", valor=P(lambda t: t in TIPOS_SERVICIO_DEUDA)),
          Faltante(nombre="vinculada"), salience=S_N1)
    def r17_falta_vinculacion(self):
        self.declare(Hecho(nombre="faltante", valor="Vinculación de la contraparte"))
        self._disparar("R17", "No se informó si la contraparte es vinculada")

    # ======================= NIVEL 2 =======================
    @Rule(Hecho(nombre="plazo_no_aplica", valor=True), salience=S_N2)
    def r18_plazo_valido_no_aplica(self):
        self.declare(Hecho(nombre="plazo_exigible_valido", valor=True))
        self._disparar("R18", "Sin exigencia de plazo para esta solicitud")

    @Rule(Hecho(nombre="plazo_minimo_requerido", valor=MATCH.minimo),
          Dato(nombre="dias_plazo", valor=MATCH.dias),
          TEST(lambda minimo, dias: dias >= minimo), salience=S_N2)
    def r19_plazo_cumple(self, minimo, dias):
        self.declare(Hecho(nombre="plazo_exigible_valido", valor=True))
        self._disparar("R19", f"{dias} días transcurridos >= {minimo} días exigidos")

    @Rule(Hecho(nombre="plazo_minimo_requerido", valor=MATCH.minimo),
          Dato(nombre="dias_plazo", valor=MATCH.dias),
          TEST(lambda minimo, dias: dias < minimo), salience=S_N2)
    def r20_plazo_no_cumple(self, minimo, dias):
        faltan = minimo - dias
        self.declare(Hecho(nombre="plazo_exigible_valido", valor=False))
        self.declare(Hecho(nombre="condicionante", valor=(
            f"Plazo de giro insuficiente: transcurrieron {dias} días y el mínimo exigido "
            f"es {minimo}; deben esperarse {faltan} días adicionales.")))
        self._disparar("R20", f"{dias} días transcurridos < {minimo} días exigidos (faltan {faltan})")

    @Rule(Hecho(nombre="inhibicion_cruzada_activa", valor=True), salience=S_N2)
    def r21_causal_bloqueante(self):
        self.declare(Hecho(nombre="causal_bloqueante_vigente", valor=True))
        self._disparar("R21", "La inhibición cruzada constituye causal impeditiva taxativa")

    @Rule(Dato(nombre="tipo_solicitud", valor="importacion_bienes"),
          Dato(nombre="categoria", valor="insumos_criticos"),
          Hecho(nombre="dj_activos_en_regla", valor=True),
          Hecho(nombre="inhibicion_cruzada_activa", valor=False), salience=S_N2)
    def r22_pago_a_la_vista(self):
        self.declare(Hecho(nombre="pago_a_la_vista_habilitado", valor=True))
        self._disparar("R22", "Insumo crítico con DJ en regla: exención de plazo de diferimiento")

    @Rule(Hecho(nombre="dj_activos_en_regla", valor=True),
          Hecho(nombre="requiere_conformidad_previa", valor=False), salience=S_N2)
    def r23_documentacion_completa(self):
        self.declare(Hecho(nombre="estado_documental", valor="completo"))
        self._disparar("R23", "DJ en regla y sin conformidad previa pendiente")

    @Rule(Hecho(nombre="dj_activos_en_regla", valor=False), salience=S_N2)
    def r24_documentacion_pendiente_fondos(self):
        self.declare(Hecho(nombre="estado_documental", valor="pendiente"))
        self.declare(Hecho(nombre="condicionante", valor=(
            f"Debe aplicar sus fondos propios líquidos del exterior (disponibilidad > USD "
            f"{_usd(UMBRAL_ACTIVOS_USD)}) o justificar su destino antes de acceder al MULC.")))
        self._disparar("R24", "Activos externos no declarados/justificados: uso de fondos propios previo")

    @Rule(Hecho(nombre="requiere_conformidad_previa", valor=True), salience=S_N2)
    def r25_documentacion_pendiente_conformidad(self):
        self.declare(Hecho(nombre="estado_documental", valor="pendiente"))
        self.declare(Hecho(nombre="condicionante", valor=(
            "Requiere conformidad previa del BCRA por tratarse de una contraparte vinculada.")))
        self._disparar("R25", "Pago a contraparte vinculada: conformidad previa del BCRA")

    # ======================= NIVEL 3 =======================
    @Rule(Hecho(nombre="causal_bloqueante_vigente", valor=True), salience=S_N3)
    def r26_situacion_impedida(self):
        self.declare(Hecho(nombre="situacion_regulatoria", valor="impedida"))
        self._disparar("R26", "Causal bloqueante vigente")

    @Rule(Hecho(nombre="inhibicion_cruzada_activa", valor=False),
          Hecho(nombre="plazo_exigible_valido", valor=False), salience=S_N3)
    def r27_condicionada_por_plazo(self):
        self.declare(Hecho(nombre="situacion_regulatoria", valor="condicionada"))
        self._disparar("R27", "Sin inhibición, pero el plazo exigible no se cumple")

    @Rule(Hecho(nombre="inhibicion_cruzada_activa", valor=False),
          Hecho(nombre="estado_documental", valor="pendiente"),
          NOT(Hecho(nombre="plazo_exigible_valido", valor=False)), salience=S_N3)
    def r28_condicionada_documental(self):
        self.declare(Hecho(nombre="situacion_regulatoria", valor="condicionada"))
        self._disparar("R28", "Sin inhibición, pero hay documentación / autorización pendiente")

    @Rule(Hecho(nombre="inhibicion_cruzada_activa", valor=False),
          Hecho(nombre="plazo_exigible_valido", valor=True),
          Hecho(nombre="estado_documental", valor="completo"), salience=S_N3)
    def r29_sin_obstaculos(self):
        self.declare(Hecho(nombre="situacion_regulatoria", valor="sin_obstaculos"))
        self._disparar("R29", "Sin inhibición, plazo válido y documentación completa")

    @Rule(Hecho(nombre="faltante", valor=MATCH.parametro),
          NOT(Hecho(nombre="situacion_regulatoria")), salience=S_INDETERMINADA)
    def r30_situacion_indeterminada(self, parametro):
        self.declare(Hecho(nombre="situacion_regulatoria", valor="indeterminada"))
        self._disparar("R30", "Con los datos disponibles no se puede determinar una situación")

    # ======================= NIVEL 4 (resolutivas) =======================
    @Rule(Hecho(nombre="situacion_regulatoria", valor="impedida"), salience=S_N4)
    def r31_dictamen_rechazado(self):
        self._cerrar(
            "RECHAZADO",
            f"Acceso al MULC bloqueado: la razón social o sus directores operaron MEP/CCL "
            f"dentro de los últimos {VENTANA_MEP_DIAS} días corridos (incompatibilidad cruzada). "
            f"La causal es impeditiva y no depende del bien o servicio.",
            ["Incompatibilidad cruzada por operaciones MEP/CCL vigente."])
        self._disparar("R31", "Dictamen final: RECHAZADO / BLOQUEADO")

    @Rule(Hecho(nombre="situacion_regulatoria", valor="condicionada"), salience=S_N4)
    def r32_dictamen_condicional(self):
        motivos = self._valores("condicionante")
        self._cerrar(
            "CONDICIONAL",
            "La operación no está bloqueada, pero requiere gestión adicional antes de cursarse.",
            motivos)
        self._disparar("R32", f"Dictamen final: CONDICIONAL ({len(motivos)} condicionante/s)")

    @Rule(Hecho(nombre="situacion_regulatoria", valor="sin_obstaculos"), salience=S_N4)
    def r33_dictamen_autorizado(self):
        a_la_vista = self._valores("pago_a_la_vista_habilitado")
        motivos = ["Sin inhibición MEP/CCL, plazo exigible cumplido y documentación completa."]
        motivos += ["Pago a la vista habilitado (insumo crítico con DJ en regla)."] * len(a_la_vista)
        self._cerrar(
            "AUTORIZADO",
            "La operación cumple la totalidad de los requisitos evaluados para cursar el pago por la entidad bancaria.",
            motivos)
        self._disparar("R33", "Dictamen final: AUTORIZADO")

    @Rule(Hecho(nombre="situacion_regulatoria", valor="indeterminada"), salience=S_N4)
    def r34_dictamen_sin_conclusion(self):
        faltantes = self._valores("faltante")
        self._cerrar(
            "SIN_CONCLUSION",
            "Información insuficiente para emitir dictamen. Complete los parámetros indicados y vuelva a evaluar.",
            [f"Falta informar: {p}" for p in faltantes])
        self._disparar("R34", f"Dictamen final: SIN CONCLUSIÓN (faltan {len(faltantes)} parámetro/s)")


# --------------------------------------------------------------------------
# API PÚBLICA
# --------------------------------------------------------------------------
def _legible(nombre, valor):
    if isinstance(valor, bool):
        return "Sí" if valor else "No"
    if nombre == "tipo_solicitud":
        return TIPOS[valor]
    if nombre == "categoria":
        return CATEGORIAS[valor]
    return str(valor)


def evaluar(tipo_solicitud=None, opero_mep=None, dias_plazo=None,
            categoria=None, activos_externos=None, vinculada=None):
    """Evalúa una solicitud. Un parámetro en None se considera NO informado (RF-06).

    Devuelve dict con: estado, etiqueta, mensaje, motivos, traza.
    """
    if tipo_solicitud is not None and tipo_solicitud not in TIPOS:
        raise ValueError(f"tipo_solicitud inválido: {tipo_solicitud!r}")
    if categoria is not None and categoria not in CATEGORIAS:
        raise ValueError(f"categoria inválida: {categoria!r}")
    if dias_plazo is not None and dias_plazo < 0:
        raise ValueError("dias_plazo no puede ser negativo")

    motor = MotorMULC()
    motor.reset()
    datos = {"tipo_solicitud": tipo_solicitud, "opero_mep": opero_mep, "dias_plazo": dias_plazo,
             "categoria": categoria, "activos_externos": activos_externos, "vinculada": vinculada}
    for nombre, valor in datos.items():
        if valor is None:
            motor.declare(Faltante(nombre=nombre))
            texto = f"{ETIQUETAS_DATOS[nombre]} = NO INFORMADO"
        else:
            motor.declare(Dato(nombre=nombre, valor=valor))
            texto = f"{ETIQUETAS_DATOS[nombre]} = {_legible(nombre, valor)}"
        motor.traza.append({
            "orden": len(motor.traza) + 1, "regla": "HECHO", "nivel": 0, "rf": "-",
            "descripcion": texto, "detalle": "", "norma": "Dato ingresado por el usuario"})
    motor.run()

    resultado = dict(motor.resultado)
    resultado["traza"] = motor.traza
    return resultado


def cantidad_de_reglas():
    return len(MotorMULC().get_rules())


def formatear_traza(traza):
    lineas = []
    for t in traza:
        sangria = "  " * t["nivel"]
        extra = f"  [{t['detalle']}]" if t["detalle"] else ""
        lineas.append(f"{t['orden']:>2}. N{t['nivel']} {sangria}{t['regla']:<5} {t['descripcion']}{extra}")
    return "\n".join(lineas)


def catalogo_markdown():
    filas = ["| ID | Nivel | RF | Regla (condición → conclusión) |", "|---|---|---|---|"]
    for rid, info in CATALOGO.items():
        filas.append(f"| {rid} | {info['nivel']} | {info['rf']} | {info['regla']} |")
    return "\n".join(filas)


if __name__ == "__main__":
    r = evaluar("importacion_bienes", False, 15, "consumo_general", False)
    print(r["etiqueta"], "-", r["mensaje"])
    print(formatear_traza(r["traza"]))

Writing motor_mulc.py


In [41]:
import importlib
import motor_mulc
importlib.reload(motor_mulc)          # permite re-ejecutar tras editar el archivo
from motor_mulc import *
from IPython.display import Markdown

print(f"Reglas en la base de conocimiento: {cantidad_de_reglas()}  (mínimo exigido: 20)")
Markdown(catalogo_markdown())

Reglas en la base de conocimiento: 34  (mínimo exigido: 20)


| ID | Nivel | RF | Regla (condición → conclusión) |
|---|---|---|---|
| R01 | 1 | RF-01 | SI operó MEP/CCL en los últimos 90 días ENTONCES inhibicion_cruzada_activa = Sí |
| R02 | 1 | RF-01 | SI NO operó MEP/CCL en los últimos 90 días ENTONCES inhibicion_cruzada_activa = No |
| R03 | 1 | RF-02 | SI importación Y categoría = Bienes de Capital ENTONCES plazo_minimo_requerido = plazo BK |
| R04 | 1 | RF-02 | SI importación Y categoría = Insumos Críticos ENTONCES plazo_minimo_requerido = a la vista |
| R05 | 1 | RF-02 | SI importación Y categoría = Consumo General ENTONCES plazo_minimo_requerido = plazo consumo |
| R06 | 1 | RF-02 | SI la solicitud es de servicios o pasivos financieros ENTONCES plazo_no_aplica = Sí |
| R07 | 1 | RF-03 | SI posee activos externos líquidos > USD 100.000 no aplicados ENTONCES dj_activos_en_regla = No |
| R08 | 1 | RF-03 | SI NO posee activos externos líquidos > USD 100.000 no aplicados ENTONCES dj_activos_en_regla = Sí |
| R09 | 1 | RF-07 | SI servicios/pasivos Y contraparte vinculada ENTONCES requiere_conformidad_previa = Sí |
| R10 | 1 | RF-07 | SI servicios/pasivos Y contraparte NO vinculada ENTONCES requiere_conformidad_previa = No |
| R11 | 1 | RF-07 | SI importación de bienes ENTONCES requiere_conformidad_previa = No (no aplica) |
| R12 | 1 | RF-06 | SI falta el tipo de solicitud ENTONCES faltante = tipo de solicitud |
| R13 | 1 | RF-06 | SI falta el historial bursátil MEP/CCL ENTONCES faltante = historial bursátil |
| R14 | 1 | RF-06 | SI importación Y falta el plazo desde registro/factura ENTONCES faltante = plazo |
| R15 | 1 | RF-06 | SI importación Y falta la categoría del bien ENTONCES faltante = categoría |
| R16 | 1 | RF-06 | SI falta la declaración jurada de activos externos ENTONCES faltante = DJ de activos externos |
| R17 | 1 | RF-06 | SI servicios/pasivos Y falta la vinculación de la contraparte ENTONCES faltante = vinculación |
| R18 | 2 | RF-02 | SI plazo_no_aplica ENTONCES plazo_exigible_valido = Sí |
| R19 | 2 | RF-02 | SI días transcurridos >= plazo_minimo_requerido ENTONCES plazo_exigible_valido = Sí |
| R20 | 2 | RF-02 | SI días transcurridos < plazo_minimo_requerido ENTONCES plazo_exigible_valido = No y se agrega condicionante de plazo |
| R21 | 2 | RF-01 | SI inhibicion_cruzada_activa ENTONCES causal_bloqueante_vigente = Sí |
| R22 | 2 | RF-02 | SI importación de insumos críticos Y DJ en regla Y sin inhibición ENTONCES pago_a_la_vista_habilitado = Sí |
| R23 | 2 | RF-03 | SI dj_activos_en_regla Y NO requiere conformidad previa ENTONCES estado_documental = completo |
| R24 | 2 | RF-03 | SI dj_activos_en_regla = No ENTONCES estado_documental = pendiente y condicionante de fondos propios |
| R25 | 2 | RF-07 | SI requiere_conformidad_previa ENTONCES estado_documental = pendiente y condicionante de conformidad BCRA |
| R26 | 3 | RF-04 | SI causal_bloqueante_vigente ENTONCES situacion_regulatoria = impedida |
| R27 | 3 | RF-04 | SI sin inhibición Y plazo_exigible_valido = No ENTONCES situacion_regulatoria = condicionada |
| R28 | 3 | RF-04 | SI sin inhibición Y estado_documental = pendiente Y plazo no incumplido ENTONCES situacion_regulatoria = condicionada |
| R29 | 3 | RF-04 | SI sin inhibición Y plazo válido Y estado_documental = completo ENTONCES situacion_regulatoria = sin obstáculos |
| R30 | 3 | RF-06 | SI hay parámetros faltantes Y no se derivó otra situación ENTONCES situacion_regulatoria = indeterminada |
| R31 | 4 | RF-01 | SI situacion_regulatoria = impedida ENTONCES ESTADO_OPERACION = RECHAZADO / BLOQUEADO |
| R32 | 4 | RF-04 | SI situacion_regulatoria = condicionada ENTONCES ESTADO_OPERACION = CONDICIONAL / REQUIERE GESTIÓN ADICIONAL |
| R33 | 4 | RF-04 | SI situacion_regulatoria = sin obstáculos ENTONCES ESTADO_OPERACION = AUTORIZADO |
| R34 | 4 | RF-06 | SI situacion_regulatoria = indeterminada ENTONCES ESTADO_OPERACION = SIN CONCLUSIÓN DETERMINABLE |

## 4. Ejemplo de ejecución con traza

Importación de bienes de consumo general a **15 días** de la nacionalización, sin operaciones MEP/CCL y sin activos externos sobre el umbral.

In [42]:
r = evaluar(tipo_solicitud="importacion_bienes", opero_mep=False, dias_plazo=15,
            categoria="consumo_general", activos_externos=False)

print("DICTAMEN :", r["etiqueta"])
print("MENSAJE  :", r["mensaje"])
for m in r["motivos"]:
    print("  -", m)
print("\nTRAZA (reglas disparadas en orden):")
print(formatear_traza(r["traza"]))

DICTAMEN : CONDICIONAL / REQUIERE GESTIÓN ADICIONAL
MENSAJE  : La operación no está bloqueada, pero requiere gestión adicional antes de cursarse.
  - Plazo de giro insuficiente: transcurrieron 15 días y el mínimo exigido es 60; deben esperarse 45 días adicionales.

TRAZA (reglas disparadas en orden):
 1. N0 HECHO Tipo de solicitud = Pago diferido de importaciones de bienes
 2. N0 HECHO Operó MEP/CCL en los últimos 90 días = No
 3. N0 HECHO Días corridos desde registro aduanero / factura = 15
 4. N0 HECHO Categoría del bien = Bienes de Consumo General
 5. N0 HECHO Disponibilidad líquida en el exterior > USD 100.000 no aplicada = No
 6. N0 HECHO Contraparte vinculada / mismo grupo económico = NO INFORMADO
 7. N1   R08   SI NO posee activos externos líquidos > USD 100.000 no aplicados ENTONCES dj_activos_en_regla = Sí  [Dato ingresado: no posee activos externos líquidos sobre el umbral]
 8. N1   R05   SI importación Y categoría = Consumo General ENTONCES plazo_minimo_requerido = plazo con

Caso incompleto: falta la categoría del bien (RF-06).

In [43]:
r = evaluar(tipo_solicitud="importacion_bienes", opero_mep=False, dias_plazo=90, activos_externos=False)
print("DICTAMEN :", r["etiqueta"])
print("MENSAJE  :", r["mensaje"])
for m in r["motivos"]:
    print("  -", m)

DICTAMEN : SIN CONCLUSIÓN DETERMINABLE / FALTAN PARÁMETROS
MENSAJE  : Información insuficiente para emitir dictamen. Complete los parámetros indicados y vuelva a evaluar.
  - Falta informar: Categoría del bien


## 5. Casos de prueba de homologación

Diez casos que incluyen los tres del brief (incompatibilidad temporal, plazos escalonados, exención crítica) y verifican, además, que la traza recorra **al menos 3 niveles** de encadenamiento. También se puede ejecutar fuera del notebook con `python casos_prueba.py`.

In [44]:
%%writefile casos_prueba.py
"""
Casos de prueba de homologación - Sistema experto MULC.

Ejecución:  python casos_prueba.py
Cada caso verifica: (1) el dictamen esperado, (2) que se disparen las reglas
clave y (3) que la traza exhiba al menos 3 niveles de encadenamiento hacia
adelante antes de la conclusión (RF-04) cuando el caso tiene datos completos.
"""
from motor_mulc import (evaluar, formatear_traza, cantidad_de_reglas,
                        ETIQUETAS_DICTAMEN)

# (id, descripción, argumentos de evaluar(), dictamen esperado, reglas que deben dispararse)
CASOS = [
    (1, "Incompatibilidad temporal: MEP en la ventana, bien de consumo general",
     dict(tipo_solicitud="importacion_bienes", opero_mep=True, dias_plazo=90,
          categoria="consumo_general", activos_externos=False),
     "RECHAZADO", ["R01", "R21", "R26", "R31"]),

    (2, "Incompatibilidad temporal independiente del bien: MEP + insumo crítico con DJ en regla",
     dict(tipo_solicitud="importacion_bienes", opero_mep=True, dias_plazo=0,
          categoria="insumos_criticos", activos_externos=False),
     "RECHAZADO", ["R01", "R21", "R26", "R31"]),

    (3, "Plazos escalonados: consumo general a 15 días de nacionalización",
     dict(tipo_solicitud="importacion_bienes", opero_mep=False, dias_plazo=15,
          categoria="consumo_general", activos_externos=False),
     "CONDICIONAL", ["R05", "R20", "R27", "R32"]),

    (4, "Exención crítica: insumo médico/sanitario con DJ en regla (pago a la vista)",
     dict(tipo_solicitud="importacion_bienes", opero_mep=False, dias_plazo=0,
          categoria="insumos_criticos", activos_externos=False),
     "AUTORIZADO", ["R04", "R19", "R22", "R23", "R29", "R33"]),

    (5, "Bien de capital con plazo cumplido (45 >= 30 días)",
     dict(tipo_solicitud="importacion_bienes", opero_mep=False, dias_plazo=45,
          categoria="bienes_capital", activos_externos=False),
     "AUTORIZADO", ["R03", "R19", "R29", "R33"]),

    (6, "Bien de capital con plazo incumplido (10 < 30 días)",
     dict(tipo_solicitud="importacion_bienes", opero_mep=False, dias_plazo=10,
          categoria="bienes_capital", activos_externos=False),
     "CONDICIONAL", ["R03", "R20", "R27", "R32"]),

    (7, "Plazo cumplido pero con activos externos líquidos sobre el umbral (fondos propios)",
     dict(tipo_solicitud="importacion_bienes", opero_mep=False, dias_plazo=90,
          categoria="consumo_general", activos_externos=True),
     "CONDICIONAL", ["R07", "R19", "R24", "R28", "R32"]),

    (8, "Pago de servicios a contraparte vinculada: conformidad previa del BCRA",
     dict(tipo_solicitud="servicios", opero_mep=False, activos_externos=False, vinculada=True),
     "CONDICIONAL", ["R09", "R25", "R28", "R32"]),

    (9, "Cancelación de pasivos financieros con contraparte no vinculada",
     dict(tipo_solicitud="pasivos_financieros", opero_mep=False, activos_externos=False, vinculada=False),
     "AUTORIZADO", ["R10", "R18", "R23", "R29", "R33"]),

    (10, "Datos incompletos: importación sin categoría del bien",
     dict(tipo_solicitud="importacion_bienes", opero_mep=False, dias_plazo=90, activos_externos=False),
     "SIN_CONCLUSION", ["R15", "R30", "R34"]),
]


def ejecutar(verbose=True):
    resultados = []
    for cid, desc, args, esperado, reglas_clave in CASOS:
        r = evaluar(**args)
        disparadas = [t["regla"] for t in r["traza"] if t["nivel"] > 0]
        niveles = sorted({t["nivel"] for t in r["traza"] if t["nivel"] > 0})
        ok_estado = r["estado"] == esperado
        faltan = [x for x in reglas_clave if x not in disparadas]
        ok_reglas = not faltan
        ok_niveles = len(niveles) >= 3
        ok = ok_estado and ok_reglas and ok_niveles
        resultados.append((cid, desc, esperado, r["estado"], niveles, faltan, ok))
        if verbose:
            marca = "OK  " if ok else "FALLA"
            print(f"[{marca}] Caso {cid:>2}: {desc}")
            print(f"         esperado={esperado}  obtenido={r['estado']}  niveles={niveles}"
                  + (f"  reglas_no_disparadas={faltan}" if faltan else ""))
    total_ok = sum(1 for x in resultados if x[-1])
    if verbose:
        print(f"\nBase de conocimiento: {cantidad_de_reglas()} reglas (mínimo exigido: 20)")
        print(f"Resultado: {total_ok}/{len(resultados)} casos aprobados")
    return total_ok == len(resultados)


if __name__ == "__main__":
    raise SystemExit(0 if ejecutar() else 1)

Overwriting casos_prueba.py


In [45]:
import casos_prueba
importlib.reload(casos_prueba)
todos_ok = casos_prueba.ejecutar()
assert todos_ok, "Hay casos de prueba que no pasaron"

[OK  ] Caso  1: Incompatibilidad temporal: MEP en la ventana, bien de consumo general
         esperado=RECHAZADO  obtenido=RECHAZADO  niveles=[1, 2, 3, 4]
[OK  ] Caso  2: Incompatibilidad temporal independiente del bien: MEP + insumo crítico con DJ en regla
         esperado=RECHAZADO  obtenido=RECHAZADO  niveles=[1, 2, 3, 4]
[OK  ] Caso  3: Plazos escalonados: consumo general a 15 días de nacionalización
         esperado=CONDICIONAL  obtenido=CONDICIONAL  niveles=[1, 2, 3, 4]
[OK  ] Caso  4: Exención crítica: insumo médico/sanitario con DJ en regla (pago a la vista)
         esperado=AUTORIZADO  obtenido=AUTORIZADO  niveles=[1, 2, 3, 4]
[OK  ] Caso  5: Bien de capital con plazo cumplido (45 >= 30 días)
         esperado=AUTORIZADO  obtenido=AUTORIZADO  niveles=[1, 2, 3, 4]
[OK  ] Caso  6: Bien de capital con plazo incumplido (10 < 30 días)
         esperado=CONDICIONAL  obtenido=CONDICIONAL  niveles=[1, 2, 3, 4]
[OK  ] Caso  7: Plazo cumplido pero con activos externos líquidos sobre

## 6. Interfaz Streamlit (RF-05)

Formulario interactivo con el dictamen destacado en color (verde / amarillo / rojo / azul), el listado ordenado de reglas activadas agrupado por nivel y el fundamento normativo de cada una. Los datos desconocidos se marcan como *"No sé / sin dato"*.

In [46]:
%%writefile app_streamlit.py
"""
Interfaz Streamlit - Sistema experto MULC (RF-05)
Ejecución (desde la terminal de VS Code, en la carpeta del proyecto):
    streamlit run app_streamlit.py
"""
import streamlit as st

from motor_mulc import (
    evaluar, cantidad_de_reglas, catalogo_markdown,
    TIPOS, CATEGORIAS, PLAZOS_MINIMOS, VENTANA_MEP_DIAS, UMBRAL_ACTIVOS_USD,
)

SIN_DATO = "No sé / sin dato"
NIVELES = {
    0: "Hechos iniciales", 1: "Hechos intermedios básicos", 2: "Hechos intermedios compuestos",
    3: "Situación regulatoria", 4: "Dictamen (regla resolutiva)",
}

st.set_page_config(page_title="Sistema experto MULC", page_icon="⚖️", layout="wide")

st.title("⚖️ Evaluación previa de acceso al MULC")
st.caption(
    "Asistente decisional analítico basado en reglas (Experta · encadenamiento hacia adelante). "
    "No constituye asesoramiento legal vinculante ni reemplaza el dictamen de la entidad bancaria."
)

# ----------------------------- Barra lateral -----------------------------
with st.sidebar:
    st.header("Parámetros normativos vigentes")
    st.write(f"**Ventana de inhibición MEP/CCL:** {VENTANA_MEP_DIAS} días corridos")
    st.write(f"**Umbral de activos externos:** USD {UMBRAL_ACTIVOS_USD:,}".replace(",", "."))
    st.write("**Plazos mínimos de diferimiento:**")
    for k, v in PLAZOS_MINIMOS.items():
        st.write(f"- {CATEGORIAS[k]}: {'a la vista' if v == 0 else f'{v} días'}")
    st.caption("Se modifican en la sección de parámetros de `motor_mulc.py`.")
    st.divider()
    st.write(f"**Base de conocimiento:** {cantidad_de_reglas()} reglas")
    with st.expander("Ver catálogo de reglas"):
        st.markdown(catalogo_markdown())


def ternario(etiqueta, clave, ayuda=None):
    opcion = st.radio(etiqueta, ["Sí", "No", SIN_DATO], index=1, horizontal=True, key=clave, help=ayuda)
    return {"Sí": True, "No": False}.get(opcion)


# ------------------------------- Formulario -------------------------------
st.subheader("1. Datos de la operación")
col1, col2 = st.columns(2)

with col1:
    tipo = st.selectbox(
        "Tipo de solicitud", [None, *TIPOS], index=1,
        format_func=lambda k: SIN_DATO if k is None else TIPOS[k])
    opero_mep = ternario(
        f"¿La razón social o sus directores vendieron valores contra moneda extranjera "
        f"(MEP/CCL) en los últimos {VENTANA_MEP_DIAS} días?", "mep")
    activos = ternario(
        f"¿Posee disponibilidad líquida en el exterior > USD {UMBRAL_ACTIVOS_USD:,} "
        f"no aplicada a operaciones permitidas?".replace(",", "."), "activos")

with col2:
    es_importacion = tipo in (None, "importacion_bienes")
    es_servicio_deuda = tipo in (None, "servicios", "pasivos_financieros")

    dias, categoria, vinculada = None, None, None
    if es_importacion:
        sin_dias = st.checkbox("No dispongo del plazo transcurrido", key="sin_dias")
        dias = None if sin_dias else int(st.number_input(
            "Días corridos desde registro aduanero / factura", min_value=0, value=15, step=1))
        categoria = st.selectbox(
            "Categoría del bien", [None, *CATEGORIAS], index=3,
            format_func=lambda k: SIN_DATO if k is None else CATEGORIAS[k])
    if es_servicio_deuda:
        vinculada = ternario("¿La contraparte es empresa del mismo grupo económico / vinculada?", "vinc")
    if not es_importacion and not es_servicio_deuda:
        st.info("No se requieren datos adicionales para este tipo de solicitud.")

evaluar_ahora = st.button("Evaluar conformidad", type="primary")

# ------------------------------- Resultado -------------------------------
if evaluar_ahora:
    r = evaluar(
        tipo_solicitud=tipo, opero_mep=opero_mep, dias_plazo=dias,
        categoria=categoria, activos_externos=activos, vinculada=vinculada)

    st.subheader("2. Dictamen")
    titulo = f"### {r['etiqueta']}"
    detalle = r["mensaje"] + "\n\n" + "\n".join(f"- {m}" for m in r["motivos"])
    {"AUTORIZADO": st.success, "CONDICIONAL": st.warning,
     "RECHAZADO": st.error, "SIN_CONCLUSION": st.info}[r["estado"]](f"{titulo}\n\n{detalle}")

    st.subheader("3. Traza de inferencia (reglas activadas, en orden)")
    niveles_usados = sorted({t["nivel"] for t in r["traza"] if t["nivel"] > 0})
    st.caption(f"Niveles de encadenamiento hacia adelante recorridos: {len(niveles_usados)} "
               f"({', '.join(f'N{n}' for n in niveles_usados)})")

    nivel_actual = None
    for t in r["traza"]:
        if t["nivel"] != nivel_actual:
            nivel_actual = t["nivel"]
            st.markdown(f"**Nivel {nivel_actual} · {NIVELES[nivel_actual]}**")
        etiqueta = "Hecho" if t["regla"] == "HECHO" else f"`{t['regla']}`"
        extra = f" — _{t['detalle']}_" if t["detalle"] else ""
        st.markdown(f"{t['orden']}. {etiqueta} · {t['descripcion']}{extra}")

    with st.expander("Fundamento normativo de cada regla disparada"):
        for t in r["traza"]:
            if t["regla"] != "HECHO":
                st.markdown(f"- `{t['regla']}` ({t['rf']}): {t['norma']}")

Writing app_streamlit.py


Para abrir la interfaz, en la **terminal de VS Code** (misma carpeta que el notebook):

```bash
streamlit run app_streamlit.py
```
Se abre en `http://localhost:8501`. (Streamlit se ejecuta fuera del notebook porque es un servidor que no termina.)

## 7. Parámetros, supuestos y límites

### Parámetros ajustables (volatilidad normativa)
Todas las premisas numéricas están aisladas al comienzo de `motor_mulc.py`: `VENTANA_MEP_DIAS`, `UMBRAL_ACTIVOS_USD` y `PLAZOS_MINIMOS`. Ante una reforma se cambia una premisa y las reglas la toman sin reentrenar ni recopilar datos. Demostración: el mismo caso (consumo general, 45 días) cambia de dictamen al modificar el plazo mínimo.

In [47]:
caso = dict(tipo_solicitud="importacion_bienes", opero_mep=False, dias_plazo=45,
            categoria="consumo_general", activos_externos=False)

original = motor_mulc.PLAZOS_MINIMOS["consumo_general"]
try:
    print(f"Plazo mínimo consumo general = {original} días ->", evaluar(**caso)["etiqueta"])
    motor_mulc.PLAZOS_MINIMOS["consumo_general"] = 30      # reforma hipotética
    print("Plazo mínimo consumo general = 30 días ->", evaluar(**caso)["etiqueta"])
finally:
    motor_mulc.PLAZOS_MINIMOS["consumo_general"] = original   # se restituye el valor

Plazo mínimo consumo general = 60 días -> CONDICIONAL / REQUIERE GESTIÓN ADICIONAL
Plazo mínimo consumo general = 30 días -> AUTORIZADO


### Supuestos que conviene validar
- **Plazos mínimos.** El brief fija la ventana de **90 días** y el umbral de **USD 100.000**, y exige que consumo general a 15 días sea condicional. Los valores concretos usados (*bienes de capital 30 días, consumo general 60 días, insumos críticos a la vista*) son **parámetros de trabajo simplificados**: hay que contrastarlos con el texto ordenado de *Exterior y Cambios* y las Com. "A" 7770, 7782, 7840 y concordantes antes de cualquier uso real. Las referencias normativas de la traza son a esas fuentes en general, no a artículos o puntos específicos.
- **Ventana MEP/CCL.** El formulario pregunta por un booleano ("¿operó en los últimos N días?"), tal como pide el brief; `VENTANA_MEP_DIAS` actualiza el texto de la pregunta y de las reglas, pero la respuesta la da el usuario.
- **Servicios y pasivos financieros.** No se les aplica el plazo ni la categoría de bienes; sí la inhibición cruzada, la DJ de activos y la conformidad previa si la contraparte es vinculada (RF-07).
- **Importaciones con contraparte vinculada.** No se evalúa la vinculación, porque RF-07 la acota a servicios y deudas.

### Decisión de diseño sobre datos ausentes
Durante las pruebas se detectó que, en Experta, detectar un dato ausente con patrones `NOT(...)` podía disparar reglas de forma intermitente según la semilla de hash de Python. Por eso los parámetros no informados se declaran explícitamente como hechos `Faltante` y las reglas R12–R17 usan solo patrones positivos. Las reglas R28 y R30 conservan un `NOT` sobre hechos derivados; el comportamiento completo se verificó con 1.296 combinaciones de entradas (incluyendo datos faltantes) contra un oráculo independiente, con resultados idénticos en 13 semillas de hash distintas.

### Fuera de alcance (según el brief)
Sin conexión a APIs de BCRA/AFIP, sin validación de autenticidad documental y sin carácter de dictamen legal vinculante.